# 04 — Feature Engineering · Databricks
**Exécuter après** : `03_unification_db.ipynb`
---

In [0]:
%pip install vaderSentiment scipy -q

In [0]:
# ============================================================
# Setup Databricks — import spark_utils par chemin absolu
# Compatible /Users/ et /Repos/ (Databricks Workspace moderne)
# ============================================================
import sys, os, importlib.util

# 1. Résolution du repo root
#    Le notebook est dans .../mon-repo/databricks/nom_notebook
#    On remonte d'un niveau par rapport au dossier 'databricks'
_ctx   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_parts = _ctx.split('/')

# Chercher le dossier 'databricks' dans le path et prendre tout ce qui est avant
_db_idx = next((i for i, p in enumerate(_parts) if p == 'databricks'), None)
if _db_idx is not None:
    _REPO = '/Workspace/' + '/'.join(_parts[1:_db_idx])
else:
    # Fallback : remonter 2 niveaux depuis le notebook
    _REPO = '/Workspace/' + '/'.join(_parts[1:-2])

_UTILS_FILE = _REPO + '/02_preprocessing/spark_utils.py'

print(f'Notebook path : {_ctx}')
print(f'Repo root     : {_REPO}')
print(f'spark_utils   : {_UTILS_FILE}')

if not os.path.exists(_UTILS_FILE):
    raise FileNotFoundError(
        f"spark_utils.py introuvable : {_UTILS_FILE}"

        f"Verifier que le repo contient bien 02_preprocessing/spark_utils.py"
    )

# 2. Import direct par chemin absolu (bypass du package pip 'spark_utils')
_spec = importlib.util.spec_from_file_location('_spark_utils', _UTILS_FILE)
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

load_raw_sources        = _mod.load_raw_sources
show_label_distribution = _mod.show_label_distribution
save_parquet            = _mod.save_parquet
load_parquet            = _mod.load_parquet
stratified_split        = _mod.stratified_split
check_class_balance     = _mod.check_class_balance

# 3. Chemins Volume Unity Catalog
# Trouver le chemin : Databricks > Data > Volumes > clic droit "fakenews" > Copy path
VOLUME        = '/Volumes/workspace/default/fakenews'  # <- MODIFIER si catalog/schema different

RAW_DIR       = VOLUME + '/raw'
PROCESSED_DIR = VOLUME + '/processed'
MODELS_DIR    = VOLUME + '/models'
COLAB_DIR     = VOLUME + '/colab_exports'
REPORTS_DIR   = VOLUME + '/reports'

for d in [PROCESSED_DIR, MODELS_DIR, COLAB_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Volume        : {VOLUME}')
print('Setup OK')

## Section 1 — Chargement des données

In [0]:
train_df = load_parquet(spark, os.path.join(PROCESSED_DIR, 'train.parquet'))
test_df  = load_parquet(spark, os.path.join(PROCESSED_DIR, 'test.parquet'))

if train_df is None or test_df is None:
    raise FileNotFoundError('train.parquet ou test.parquet introuvable. Executer 03_unification_db.ipynb.')

print(f'Train : {train_df.count():,} lignes')
print(f'Test  : {test_df.count():,} lignes')
display(train_df.limit(3))

## Section 2 — TF-IDF (scikit-learn)

In [0]:
# TF-IDF via scikit-learn — compatible cluster Shared
# (RegexTokenizer / HashingTF / IDF Spark ML bloqués par le security manager)
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import scipy.sparse as sp
import joblib

print('Collecte des textes Spark -> pandas...')
train_pd = train_df.select('clean_text', 'label').toPandas()
test_pd  = test_df.select('clean_text',  'label').toPandas()
train_pd['clean_text'] = train_pd['clean_text'].fillna('')
test_pd['clean_text']  = test_pd['clean_text'].fillna('')
print(f'Train : {len(train_pd):,} | Test : {len(test_pd):,}')

print('Entrainement TF-IDF sklearn...')
tfidf_vec = TfidfVectorizer(
    max_features=50_000,
    min_df=2,
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'[a-z]{3,}'
)
X_train = tfidf_vec.fit_transform(train_pd['clean_text'])
X_test  = tfidf_vec.transform(test_pd['clean_text'])
y_train = train_pd['label'].values
y_test  = test_pd['label'].values
print(f'TF-IDF OK : X_train={X_train.shape} | X_test={X_test.shape}')

# Sauvegarder le vectoriseur
os.makedirs(os.path.join(MODELS_DIR, 'baseline'), exist_ok=True)
tfidf_pkl = os.path.join(MODELS_DIR, 'baseline', 'tfidf_vectorizer.pkl')
joblib.dump(tfidf_vec, tfidf_pkl)
print(f'Vectoriseur sauvegarde : {tfidf_pkl}')

## Section 3 — Features NLP (VADER Sentiment)

In [0]:
# Features NLP calculees dans 06_nlp_classical_db.ipynb (pandas, pas de Spark UDF)
print("Features NLP : voir 06_nlp_classical_db.ipynb")

## Section 4 — Export pour Google Colab (DistilBERT)

In [0]:
# Export pour Google Colab et notebooks modeling suivants
import shutil, os

# Écrire d'abord dans /tmp/ (Unity Catalog Volumes ne supportent pas les écritures binaires directes)
TMP = '/tmp/fakenews_exports'
os.makedirs(TMP, exist_ok=True)

print('Sauvegarde des features dans /tmp/...')
sp.save_npz(os.path.join(TMP, 'train_features.npz'), X_train)
sp.save_npz(os.path.join(TMP, 'test_features.npz'),  X_test)
np.save(os.path.join(TMP, 'train_labels.npy'), y_train)
np.save(os.path.join(TMP, 'test_labels.npy'),  y_test)

train_pd[['clean_text','label']].to_csv(os.path.join(TMP, 'train_texts.csv'), index=False)
test_pd[['clean_text','label']].to_csv(os.path.join(TMP,  'test_texts.csv'),  index=False)

# Copier vers le Volume
print(f'Copie vers {COLAB_DIR}...')
os.makedirs(COLAB_DIR, exist_ok=True)
for fname in os.listdir(TMP):
    src  = os.path.join(TMP, fname)
    dest = os.path.join(COLAB_DIR, fname)
    shutil.copy2(src, dest)
    print(f'  {fname}')

print(f'\nFeatures    : X_train={X_train.shape} | X_test={X_test.shape}')
print(f'Textes      : {len(train_pd):,} train | {len(test_pd):,} test')
print(f'Repertoire  : {COLAB_DIR}')
print('Prochaine etape : 05_baseline_tfidf_db.ipynb')

In [0]:
# Vérification des fichiers dans le Volume
print(f'Fichiers exportes dans {COLAB_DIR} :')
import os
for fname in sorted(os.listdir(COLAB_DIR)):
    size_mb = os.path.getsize(os.path.join(COLAB_DIR, fname)) / 1024 / 1024
    print(f'  {fname:<35} {size_mb:.1f} MB')

print('Prochaine etape : 05_baseline_tfidf_db.ipynb')